# 3. Biological analysis and interpretation

This notebook uses bundled published embeddings and curated cell-type metadata. It demonstrates quality control, six-group alignment, standardized projection/clustering, deterministic classification, group comparison, neighbor context, and MoBIE-compatible export. These are exploratory and predictive analyses: neither a cluster nor classification accuracy alone establishes a biological mechanism.

In [ ]:
from pathlib import Path
import importlib.util
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from morphofeatures.analysis.classification import (
    cross_validate_logistic, load_class_labels, select_labeled_embeddings
)
from morphofeatures.analysis.context import aggregate_neighbors
from morphofeatures.analysis.projection import cluster_embeddings, compute_umap
from morphofeatures.config import load_config
from morphofeatures.data.contracts import EmbeddingTable
from morphofeatures.data.io import export_embeddings, load_embeddings, merge_embeddings

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter from the MorphoFeatures repository root')
config = load_config()
output_dir = config.paths.output_root / 'notebooks' / '03_biological_analysis'
output_dir.mkdir(parents=True, exist_ok=True)
published = load_embeddings(config.paths.analysis_data / 'morphofeatures_all_cells.npy', mmap=True)
{'cells': len(published.label_ids), 'features': published.features.shape[1],
 'finite': bool(np.isfinite(published.features).all()),
 'unique_labels': len(np.unique(published.label_ids)) == len(published.label_ids)}

## Align and combine the six 80-dimensional groups

Each group is independently trained. `merge_embeddings` sorts and requires identical IDs, so this is an ID join rather than a positional concatenation.

In [ ]:
group_names = [
    'features_shape_cell.tsv', 'features_shape_nucl.tsv',
    'features_coarse_ultr_cell.tsv', 'features_coarse_ultr_nucl.tsv',
    'features_fine_ultr_cell.tsv', 'features_fine_ultr_nucl.tsv',
]
group_paths = [config.paths.mobie_data / name for name in group_names]
groups = [load_embeddings(path) for path in group_paths]
combined = merge_embeddings(group_paths)
assert combined.features.shape[1] == 6 * 80
assert np.array_equal(combined.label_ids, published.sorted().label_ids)
pd.DataFrame({'group': group_names, 'cells': [len(group.label_ids) for group in groups],
              'features': [group.features.shape[1] for group in groups]})

## Standardized projection and clustering

Feature standardization prevents dimensions with larger numeric scales from dominating distances. UMAP is used when the analysis extra is installed; otherwise the cell executes a clearly labeled PCA fallback so the notebook remains runnable. K-means is deterministic here and is not presented as a cell-type definition.

In [ ]:
rng = np.random.default_rng(42)
selected = np.sort(rng.choice(len(combined.label_ids), size=1000, replace=False))
selected_ids = combined.label_ids[selected]
scaled = StandardScaler().fit_transform(combined.features[selected])
coordinates = None
if importlib.util.find_spec('umap') is not None:
    try:
        coordinates = compute_umap(scaled, n_neighbors=15, seed=42, n_epochs=50)
        projection_method = 'UMAP'
    except (RuntimeError, OSError) as error:
        print(f'UMAP runtime unavailable ({error}); using PCA fallback.')
if coordinates is None:
    coordinates = PCA(n_components=2, random_state=42).fit_transform(scaled)
    projection_method = 'PCA fallback (install analysis extra for UMAP)'
clusters = cluster_embeddings(scaled, method='kmeans', n_clusters=8, seed=42)
projection_frame = pd.DataFrame({
    'label_id': selected_ids, 'cluster': clusters,
    'axis_1': coordinates[:, 0], 'axis_2': coordinates[:, 1],
})
class_frame = pd.read_csv(config.paths.analysis_data / 'class_labels.tsv', sep='\t')
projection_frame = projection_frame.merge(
    class_frame[['label_id', 'cell_type']], on='label_id', how='left', validate='one_to_one'
)
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(projection_frame.axis_1, projection_frame.axis_2,
                     c=projection_frame.cluster, s=12, cmap='tab10', alpha=0.75)
ax.set(title=f'{projection_method}: exploratory structure', xlabel='axis 1', ylabel='axis 2')
fig.colorbar(scatter, ax=ax, label='K-means cluster');

The plot can reveal neighborhoods and outliers worth checking against anatomy and image quality. It cannot prove that clusters are discrete biological types: UMAP/PCA and K-means depend on preprocessing and parameters, and only a subset has curated annotations.

In [ ]:
label_ids, labels, class_names = load_class_labels(config.paths.analysis_data / 'class_labels.tsv')
features, labels = select_labeled_embeddings(combined, label_ids, labels)
features = StandardScaler().fit_transform(features)
classification = cross_validate_logistic(
    features, labels, class_names, folds=3, seed=42, max_iter=1000
)
per_class_recall = np.divide(
    np.diag(classification.confusion), classification.confusion.sum(axis=1),
    out=np.zeros(len(class_names), dtype=float),
    where=classification.confusion.sum(axis=1) > 0,
)
display(pd.DataFrame({'cell_type': class_names, 'cross_validated_recall': per_class_recall}))
pd.DataFrame(classification.confusion, index=class_names, columns=class_names).style.background_gradient(cmap='Blues')

Cross-validated accuracy and confusion summarize predictive separability for this annotation set. They do not imply causal morphology–identity relationships, and class imbalance, annotation choices, specimen sampling, and shared preprocessing can affect estimates.

In [ ]:
# A controlled group comparison: identical labels, split seed, model, and scaling.
comparison_tables = {
    'cell shape': groups[0], 'coarse cell texture': groups[2],
    'fine cell texture': groups[4], 'all six groups': combined,
}
comparison = []
for name, table in comparison_tables.items():
    values, same_labels = select_labeled_embeddings(table, label_ids, labels)
    result = cross_validate_logistic(
        StandardScaler().fit_transform(values), same_labels, class_names,
        folds=3, seed=42, max_iter=1000,
    )
    comparison.append({'representation': name, 'dimensions': values.shape[1],
                       'mean_accuracy': result.mean_accuracy, 'std': result.std_accuracy})
comparison_frame = pd.DataFrame(comparison).sort_values('mean_accuracy', ascending=False)
display(comparison_frame)
comparison_frame.set_index('representation').mean_accuracy.plot.bar(
    ylim=(0, 1), ylabel='3-fold accuracy', title='Predictive group comparison (same protocol)'
)

Different group accuracy suggests differential predictive information under this model; it does not rank biological importance, and the groups may differ in noise, preprocessing, or redundancy. MAE and contrastive training losses are not directly comparable—compare held-out downstream behavior and anatomical consistency.

In [ ]:
# Demonstrate neighbor aggregation on a bounded subset and compare published context coverage.
with (config.paths.analysis_data / 'bilateral_neighbors.pkl').open('rb') as stream:
    neighbor_mapping = pickle.load(stream)
subset_table = EmbeddingTable(combined.label_ids[:500], combined.features[:500])
context_subset = aggregate_neighbors(subset_table, neighbor_mapping, include_self=True, reducer='mean')
published_context = load_embeddings(
    config.paths.analysis_data / 'morphocontextfeatures_all_cells_agglomerated.npy'
)
coverage = {
    'aggregated_subset_cells': len(context_subset.label_ids),
    'published_context_cells': len(published_context.label_ids),
    'published_context_features': published_context.features.shape[1],
    'published_context_ids_in_morphofeatures': len(
        set(published_context.label_ids).intersection(combined.label_ids)
    ),
}
coverage

In [ ]:
# MoBIE-compatible outputs keep label_id first. Joins above and below are always on label_id.
projection_output = export_embeddings(
    output_dir / 'projection_clusters.tsv',
    projection_frame.label_id,
    projection_frame[['cluster', 'axis_1', 'axis_2']].to_numpy(),
    ('cluster', 'projection_1', 'projection_2'),
)
context_output = export_embeddings(
    output_dir / 'context_subset.npy', context_subset.label_ids, context_subset.features
)
{'mobie_table': projection_output, 'context_embedding': context_output}

## Interpretation boundary

Projection shows parameter-dependent exploratory structure; clustering proposes partitions; classification estimates prediction on annotated examples; context summarizes selected neighbors. Biological evidence requires returning to registered imagery, checking segmentation/QC and spatial/anatomical covariates, independent specimens, and domain validation. The bundled arrays reproduce downstream software analyses, while full historical model retraining still requires external raw data, exact preprocessing/configuration, and original checkpoints.